# Nassau Candy Distributor
## Product Line Profitability & Margin Performance Analysis
This notebook reproduces the core EDA, cleaning, product/division profitability metrics, Pareto analysis, and margin-risk diagnostics.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
ROOT=Path('.')
df=pd.read_csv(ROOT/'data'/'raw_nassau_candy.csv')
df.head()

## 1. Data quality checks

In [ ]:
df.shape, df.isna().sum(), df.duplicated().sum()

In [ ]:
for c in ['Order Date','Ship Date']:
    df[c]=pd.to_datetime(df[c],dayfirst=True,errors='coerce')
print('Order dates:',df['Order Date'].min(),df['Order Date'].max())
print('Ship dates:',df['Ship Date'].min(),df['Ship Date'].max())
print('Max reconciliation error:',(df['Sales']-df['Cost']-df['Gross Profit']).abs().max())

**Caveat:** Ship dates are inconsistent with the order period, so shipping-duration conclusions are excluded.

## 2. KPI engineering

In [ ]:
df['Gross Margin']=df['Gross Profit']/df['Sales']
df['Profit per Unit']=df['Gross Profit']/df['Units']
print('Sales',df.Sales.sum())
print('Gross Profit',df['Gross Profit'].sum())
print('Gross Margin',df['Gross Profit'].sum()/df.Sales.sum())

## 3. Product profitability

In [ ]:
p=df.groupby(['Product ID','Product Name','Division'],as_index=False).agg(Sales=('Sales','sum'),Cost=('Cost','sum'),Gross_Profit=('Gross Profit','sum'),Units=('Units','sum'))
p['Gross_Margin']=p.Gross_Profit/p.Sales
p['Profit_per_Unit']=p.Gross_Profit/p.Units
p['Revenue_Contribution']=p.Sales/p.Sales.sum()
p['Profit_Contribution']=p.Gross_Profit/p.Gross_Profit.sum()
p.sort_values('Gross_Profit',ascending=False)

## 4. Division performance

In [ ]:
d=df.groupby('Division',as_index=False).agg(Sales=('Sales','sum'),Cost=('Cost','sum'),Gross_Profit=('Gross Profit','sum'),Units=('Units','sum'))
d['Gross_Margin']=d.Gross_Profit/d.Sales
d['Revenue_Share']=d.Sales/d.Sales.sum()
d['Profit_Share']=d.Gross_Profit/d.Gross_Profit.sum()
d

## 5. Pareto concentration

In [ ]:
for metric in ['Sales','Gross_Profit']:
    t=p.sort_values(metric,ascending=False).copy()
    t['cum']=t[metric].cumsum()/t[metric].sum()
    n=(t['cum']<.8).sum()+1
    print(metric,':',n,'of',len(t),'products cross 80%')

## 6. Margin risk diagnostics

In [ ]:
p['Risk']=np.select([p.Gross_Margin<.20,p.Gross_Margin<.50,p.Gross_Margin<.60],['Critical','High','Watch'],default='Healthy')
p.sort_values('Gross_Margin')[['Product Name','Sales','Gross_Profit','Gross_Margin','Profit_per_Unit','Risk']]

## 7. Business interpretation
- Protect the five dominant chocolate products.
- Put Kazookles into immediate repricing/cost/discontinuation review.
- Review Lickable Wallpaper pricing/cost architecture.
- Treat Sugar primarily as a scale/growth question, not a broad margin-repair problem.
- Add cost-to-serve fields before making contribution-margin decisions.